In [19]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score
)


In [20]:

df_sample = pd.read_csv("data/df_sample.csv")
print(df_sample.shape)
df_sample.head()


(200000, 75)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,target_bad
0,1077501,1296599,5000.0,5000.0,4975.0,36 months,10.65,162.87,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,1077430,1314167,2500.0,2500.0,2500.0,60 months,15.27,59.83,C,C4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2,1077175,1313524,2400.0,2400.0,2400.0,36 months,15.96,84.33,C,C5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1076863,1277178,10000.0,10000.0,10000.0,36 months,13.49,339.31,C,C1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,1075358,1311748,3000.0,3000.0,3000.0,60 months,12.69,67.79,B,B5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Celda 1. Definir target_col y listas de columnas

In [21]:
# 1) Target
if "target_bad" in df_sample.columns:
    target_col = "target_bad"
else:
    raise NameError(
        "No existe target_bad en df_sample. Regresa a 01_eda, crea target_bad y vuelve a guardar data/df_sample.csv"
    )

# 2) Columnas que no deben entrar al modelo
id_text_cols = []
for c in ["id", "member_id", "url", "desc", "emp_title", "title"]:
    if c in df_sample.columns:
        id_text_cols.append(c)

leak_candidates = [
    "loan_status",
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "out_prncp", "out_prncp_inv",
]

leak_cols = [c for c in leak_candidates if c in df_sample.columns]

optional_cols = []
base_cols = [c for c in df_sample.columns if c not in ([target_col] + id_text_cols + leak_cols)]
X_cols = base_cols + optional_cols

print("target_col:", target_col)
print("id_text_cols:", len(id_text_cols), id_text_cols)
print("leak_cols:", len(leak_cols), leak_cols)
print("X_cols:", len(X_cols))

df_sample[target_col].value_counts(dropna=False)


target_col: target_bad
id_text_cols: 6 ['id', 'member_id', 'url', 'desc', 'emp_title', 'title']
leak_cols: 14 ['loan_status', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'out_prncp', 'out_prncp_inv']
X_cols: 54


target_bad
0.0    106186
NaN     66619
1.0     27195
Name: count, dtype: int64

# Celda 2. Split

In [22]:
RANDOM_STATE = 12345

# Dataset para modelar sin NaN en target
df_model = df_sample.dropna(subset=[target_col]).copy()

# Detecta columnas totalmente vacías
cols_all_nan = [c for c in X_cols if df_model[c].isna().all()]
print("Columnas 100% NaN:", len(cols_all_nan))
print(cols_all_nan)

# Actualiza lista de features
X_cols_clean = [c for c in X_cols if c not in cols_all_nan]
print("X_cols antes:", len(X_cols))
print("X_cols después:", len(X_cols_clean))

# Split
X = df_model[X_cols_clean].copy()
y = df_model[target_col].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Distribución y_train")
print(y_train.value_counts(normalize=True).round(4))

Columnas 100% NaN: 17
['annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_il_6m', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m']
X_cols antes: 54
X_cols después: 37
Train: (106704, 37) Test: (26677, 37)
Distribución y_train
target_bad
0    0.7961
1    0.2039
Name: proportion, dtype: float64


# Preprocesamiento y modelo

In [23]:
# Preprocesamiento y modelo

num_features = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]

print("Num features:", len(num_features))
print("Cat features:", len(cat_features))

num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features)
    ],
    remainder="drop"
)

model = LogisticRegression(
    max_iter=6000,
    tol=1e-3,
    solver="saga",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model)
    ]
)

clf.fit(X_train, y_train)

proba_test = clf.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

print("Listo, modelo entrenado")


Num features: 23
Cat features: 14


/Users/mauvilar/.venvs/credit_risk_311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Listo, modelo entrenado


# Métricas y lectura de riesgo

In [24]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score
)

cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

recall_1 = recall_score(y_test, pred_test, pos_label=1)
precision_1 = precision_score(y_test, pred_test, pos_label=1, zero_division=0)
roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

approved = (pred_test == 0).sum()
approval_rate = approved / len(pred_test)

bad_approved = fn
bad_approval_rate_among_approved = bad_approved / max(approved, 1)

print("Confusion matrix [0,1]")
print(cm)

print("\nMétricas clave")
print("Recall clase 1:", round(recall_1, 4))
print("Precision clase 1:", round(precision_1, 4))
print("ROC AUC:", round(roc_auc, 4))
print("PR AUC:", round(pr_auc, 4))

print("\nNegocio")
print("Approval rate:", round(approval_rate, 4))
print("Bad approvals (FN):", int(bad_approved))
print("Bad approval rate among approved:", round(bad_approval_rate_among_approved, 4))

print("\nReporte")
print(classification_report(y_test, pred_test, digits=4))


Confusion matrix [0,1]
[[13783  7455]
 [ 1861  3578]]

Métricas clave
Recall clase 1: 0.6578
Precision clase 1: 0.3243
ROC AUC: 0.7085
PR AUC: 0.3725

Negocio
Approval rate: 0.5864
Bad approvals (FN): 1861
Bad approval rate among approved: 0.119

Reporte
              precision    recall  f1-score   support

           0     0.8810    0.6490    0.7474     21238
           1     0.3243    0.6578    0.4344      5439

    accuracy                         0.6508     26677
   macro avg     0.6027    0.6534    0.5909     26677
weighted avg     0.7675    0.6508    0.6836     26677



# Tabla de umbrales para bajar FN y controlar aprobación

In [25]:
import pandas as pd

def metrics_at_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    approved = (pred == 0).sum()
    total = len(pred)

    return {
        "threshold": float(thr),
        "recall_1": float(recall_score(y_true, pred, pos_label=1)),
        "precision_1": float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
        "approval_rate": float(approved / total),
        "bad_approved": int(fn),
        "bad_approval_rate_among_approved": float(fn / max(approved, 1))
    }

thresholds = np.round(np.linspace(0.1, 0.9, 17), 2)
thr_table = pd.DataFrame([metrics_at_threshold(y_test, proba_test, t) for t in thresholds])

thr_table.sort_values(["bad_approved", "approval_rate"], ascending=[True, False]).head(10)


,threshold,recall_1,precision_1,approval_rate,bad_approved,bad_approval_rate_among_approved
0,0.10,0.997058,0.207064,0.018255,16,0.032854
1,0.15,0.990623,0.213395,0.053529,51,0.035714
2,0.20,0.976650,0.221435,0.100761,127,0.047247
3,0.25,0.957897,0.232278,0.159201,229,0.053920
4,0.30,0.929031,0.245972,0.229936,386,0.062928
5,0.35,0.879757,0.260706,0.311992,654,0.078577
6,0.40,0.819820,0.279614,0.402219,980,0.091333
7,0.45,0.737636,0.298801,0.496683,1427,0.107698
8,0.50,0.657842,0.324300,0.586423,1861,0.118959
9,0.55,0.561133,0.349880,0.673014,2387,0.132951


# Elegir umbral con regla de negocio

In [26]:
min_approval = 0.50

cand = thr_table[thr_table["approval_rate"] >= min_approval].copy()
cand = cand.sort_values(["bad_approved", "recall_1"], ascending=[True, False])

print("Candidatos:", cand.shape[0])
cand.head(10)


Candidatos: 9


,threshold,recall_1,precision_1,approval_rate,bad_approved,bad_approval_rate_among_approved
8,0.50,0.657842,0.324300,0.586423,1861,0.118959
9,0.55,0.561133,0.349880,0.673014,2387,0.132951
10,0.60,0.459459,0.375169,0.750309,2940,0.146882
11,0.65,0.352454,0.403664,0.821981,3522,0.160617
12,0.70,0.245082,0.432652,0.884507,4106,0.174013
13,0.75,0.151866,0.475806,0.934925,4613,0.184956
14,0.80,0.077588,0.523573,0.969787,5017,0.193924
15,0.85,0.027946,0.573585,0.990066,5287,0.200174
16,0.90,0.003677,0.526316,0.998576,5419,0.203424
